# Validation

How do we know the estimator is correct and not 'a big pile of code that
returns nonsense'? Two independent checks.

1. **Trivial limit:** with thresholds pushed beyond the data range,
   nothing is censored, so the censored/truncated MLE *must* reduce to
   ordinary least squares. We verify this against `statsmodels.OLS`.
2. **Reference packages:** the test suite also compares against R's
   `AER::tobit` and `truncreg` (run when R is available); see
   `tests/test_r_reference.py`.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from censtrunc import CensoredRegression, TruncatedRegression

rng = np.random.default_rng(0)
n = 1000
X = rng.normal(size=(n, 3))
y = 2.0 + 1.5*X[:,0] - 0.7*X[:,1] + 0.3*X[:,2] + rng.normal(scale=1.3, size=n)

## No censoring $\Rightarrow$ OLS

We set `left=-1e6`, `right=1e6` so that no observation is censored. The censored log-likelihood then reduces to the Gaussian log-likelihood, whose maximiser is exactly the OLS coefficient vector.

In [2]:
ols = sm.OLS(y, sm.add_constant(X)).fit()
cens = CensoredRegression(left=-1e6, right=1e6).fit(X, y)
trunc = TruncatedRegression(left=-1e6, right=1e6).fit(X, y)

pd.DataFrame({
    'OLS':            np.asarray(ols.params),
    'Censored MLE':   cens.coef_,
    'Truncated MLE':  trunc.coef_,
}, index=['const', 'x1', 'x2', 'x3']).round(6)

,OLS,Censored MLE,Truncated MLE
const,2.046512,2.046482,2.046512
x1,1.434191,1.434221,1.434191
x2,-0.626598,-0.626603,-0.626598
x3,0.345980,0.345982,0.345980


The columns agree to several decimals. We can quantify the maximum discrepancy and confirm the scale parameter and log-likelihood match too (using the MLE scale $\hat\sigma = \sqrt{\mathrm{SSR}/n}$).

In [3]:
ols_beta = np.asarray(ols.params)
print(f'max |beta_censored - beta_OLS|  = {np.max(np.abs(cens.coef_ - ols_beta)):.2e}')
print(f'max |beta_truncated - beta_OLS| = {np.max(np.abs(trunc.coef_ - ols_beta)):.2e}')
print(f'|sigma_censored - sqrt(SSR/n)|  = {abs(cens.sigma_ - np.sqrt(ols.ssr/n)):.2e}')
print(f'|loglik_censored - loglik_OLS|  = {abs(cens.llf_ - ols.llf):.2e}')

max |beta_censored - beta_OLS|  = 3.00e-05
max |beta_truncated - beta_OLS| = 4.44e-16
|sigma_censored - sqrt(SSR/n)|  = 7.55e-07
|loglik_censored - loglik_OLS|  = 5.90e-07


All discrepancies are at the level of the optimiser tolerance — the estimator behaves exactly as theory requires in the no-censoring limit, which is strong evidence the likelihood and its optimisation are implemented correctly.